# PEDALI — Partie 3 : Spark SQL, DataFrames et Parquet
**Jalon 1 — Écosystème Big Data 2, EMSI**  
Ce notebook utilise le **cluster Spark 3.5.3 distribué** (`spark://spark-master:7077`) et les **50 000 événements** des parties précédentes.

**Ordre :** Kernel → Restart Kernel, puis **Run All** jusqu'à la section « MinIO » (cette dernière demande les identifiants locaux).  
**Prérequis :** les trois conteneurs Spark, le conteneur Jupyter et MinIO doivent être actifs ; `/workspace` doit être monté dans les Workers et Jupyter.

**Preuves pour le rapport :** P3-01 calculs DataFrame/SQL et mesures ; P3-02 jointure et `explain(True)` ; P3-03 fichiers Parquet dans MinIO.

## 1. SparkSession distribuée — ne pas utiliser `local[*]`

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               DoubleType, LongType)
from pathlib import Path
import time, json, os

# Après un Restart Kernel, la session est créée sur le cluster Docker.
spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("PEDALI-Partie3-DataFrame-SQL")
    .config("spark.driver.host", "pedali-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Version     :", spark.version)
print("Master      :", sc.master)
print("Application :", sc.appName)
print("Parallelisme:", sc.defaultParallelism)
print("Spark UI    :", sc.uiWebUrl)
assert spark.version == "3.5.3"
assert sc.master == "spark://spark-master:7077", "Restart Kernel puis relancer cette cellule"
print("VALIDATION : CLUSTER DISTRIBUE OK")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 12:59:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version     : 3.5.3
Master      : spark://spark-master:7077
Application : PEDALI-Partie3-DataFrame-SQL
Parallelisme: 2
Spark UI    : http://pedali-jupyter:4040
VALIDATION : CLUSTER DISTRIBUE OK


## 2. Charger les 50 000 trajets avec un schéma explicite

In [2]:

sc = spark.sparkContext

print("=== TEST CLUSTER PEDALI PARTIE 3 ===")
print("Spark :", spark.version)
print("Master :", sc.master)
print("Application :", sc.appName)

resultat = sc.parallelize([10, 20, 30, 40], 2).sum()

print("Resultat :", resultat)

assert sc.master == "spark://spark-master:7077"
assert resultat == 100

print("PARTIE 3 : CLUSTER DISTRIBUE OK")


=== TEST CLUSTER PEDALI PARTIE 3 ===
Spark : 3.5.3
Master : spark://spark-master:7077
Application : PEDALI-Partie3-DataFrame-SQL


[Stage 0:>                                                          (0 + 2) / 2]

Resultat : 100
PARTIE 3 : CLUSTER DISTRIBUE OK


In [3]:
TRIPS_FILE = "file:///workspace/data/bronze/trips_history.jsonl"
STATIONS_FILE = Path("/workspace/data/stations.json")

schema_trips = StructType([
    StructField("trip_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("vehicle_type", StringType(), True),
    StructField("start_station", StringType(), True),
    StructField("end_station", StringType(), True),
    StructField("city", StringType(), True),
    StructField("status", StringType(), True),
    StructField("duration_s", IntegerType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("battery_pct", IntegerType(), True),
    StructField("price_mad", DoubleType(), True),
    StructField("event_time", StringType(), True),
])

df_trips = spark.read.schema(schema_trips).json(TRIPS_FILE)
df_trips.createOrReplaceTempView("trips")
print("Nombre de lignes :", df_trips.count())
print("Nombre de colonnes :", len(df_trips.columns))
df_trips.printSchema()
df_trips.show(5, truncate=False)
assert df_trips.count() == 50000
assert len(df_trips.columns) == 13

Nombre de lignes : 50000
Nombre de colonnes : 13
root
 |-- trip_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- start_station: string (nullable = true)
 |-- end_station: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- duration_s: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- battery_pct: integer (nullable = true)
 |-- price_mad: double (nullable = true)
 |-- event_time: string (nullable = true)

+----------+-------+----------+------------+-------------+-----------+----------+--------+----------+-----------+-----------+---------+--------------------+
|trip_id   |user_id|vehicle_id|vehicle_type|start_station|end_station|city      |status  |duration_s|distance_km|battery_pct|price_mad|event_time          |
+----------+-------+----------+------------+-------------+-----------+----------+--------+--

## 3. Chiffre d’affaires : DataFrame puis SQL (comparaison avec RDD)

In [4]:
df_ended = df_trips.filter((F.col("status") == "ENDED") & F.col("price_mad").isNotNull())
print("Lignes ENDED facturables :", df_ended.count())
assert df_ended.count() == 30228

# API DataFrame
ca_df = (df_ended.groupBy("city")
         .agg(F.round(F.sum("price_mad"), 2).alias("ca_mad"))
         .orderBy(F.desc("ca_mad")))

# API Spark SQL
ca_sql = spark.sql("""
SELECT city, ROUND(SUM(price_mad), 2) AS ca_mad
FROM trips
WHERE status = 'ENDED' AND price_mad IS NOT NULL
GROUP BY city
ORDER BY ca_mad DESC
""")

print("=== DATAFRAME ===")
ca_df.show(truncate=False)
print("=== SPARK SQL ===")
ca_sql.show(truncate=False)

attendu_rdd = {"Casablanca": 174698.09, "Marrakech": 173034.32,
              "Tanger": 172358.22, "Rabat": 172312.90}
res_df = {r['city']: float(r['ca_mad']) for r in ca_df.collect()}
res_sql = {r['city']: float(r['ca_mad']) for r in ca_sql.collect()}
assert res_df == res_sql
for ville, montant in attendu_rdd.items():
    assert abs(res_df[ville] - montant) < 0.02, (ville, res_df[ville], montant)
print("CA TOTAL :", round(sum(res_df.values()), 2), "MAD")
print("VALIDATION : RDD = DataFrame = SQL (a l'arrondi pres)")

Lignes ENDED facturables : 30228
=== DATAFRAME ===


+----------+---------+
|city      |ca_mad   |
+----------+---------+
|Casablanca|174698.09|
|Marrakech |173034.32|
|Tanger    |172358.22|
|Rabat     |172312.9 |
+----------+---------+

=== SPARK SQL ===
+----------+---------+
|city      |ca_mad   |
+----------+---------+
|Casablanca|174698.09|
|Marrakech |173034.32|
|Tanger    |172358.22|
|Rabat     |172312.9 |
+----------+---------+

CA TOTAL : 692403.53 MAD
VALIDATION : RDD = DataFrame = SQL (a l'arrondi pres)


### Mesure de temps (répétitions, pas une seule exécution)

In [5]:
import statistics

def chrono(action, repetitions=3):
    action()  # échauffement (JVM et caches)
    temps = []
    for _ in range(repetitions):
        t0 = time.perf_counter()
        action()
        temps.append(time.perf_counter() - t0)
    return temps

mesures_df = chrono(lambda: ca_df.collect())
mesures_sql = chrono(lambda: ca_sql.collect())
print("DataFrame (s) :", [round(x, 3) for x in mesures_df],
      "mediane =", round(statistics.median(mesures_df), 3))
print("SQL       (s) :", [round(x, 3) for x in mesures_sql],
      "mediane =", round(statistics.median(mesures_sql), 3))
print("Note : ces mesures portent sur le meme cluster ; ne pas conclure a une supériorité universelle.")

DataFrame (s) : [0.111, 0.097, 0.109] mediane = 0.109
SQL       (s) : [0.15, 0.09, 0.13] mediane = 0.13
Note : ces mesures portent sur le meme cluster ; ne pas conclure a une supériorité universelle.


## 4. Stations : schéma explicite et jointure

Le fichier `stations.json` doit avoir 400 stations. Les noms des champs peuvent varier selon le générateur. On choisit leurs noms **à partir du JSON brut**, puis on construit un `StructType` explicite (aucune inférence de schéma par Spark).

In [6]:
assert STATIONS_FILE.exists(), f"Absent : {STATIONS_FILE}"
texte_stations = STATIONS_FILE.read_text(encoding="utf-8-sig").strip()
# Gère un tableau JSON ou un JSON par ligne.
if texte_stations.startswith("["):
    stations_obj = json.loads(texte_stations)
    multiligne = True
elif texte_stations.startswith("{"):
    try:
        obj = json.loads(texte_stations)
        stations_obj = ([obj] if any(not isinstance(v, list) for v in obj.values())
                        else next((v for v in obj.values() if isinstance(v, list)), [obj]))
        multiligne = True
    except json.JSONDecodeError:
        stations_obj = [json.loads(l) for l in texte_stations.splitlines() if l.strip()]
        multiligne = False
else:
    raise ValueError("Format de stations.json non reconnu")
assert stations_obj and isinstance(stations_obj[0], dict)
cles = set().union(*(s.keys() for s in stations_obj))
print("Champs stations :", sorted(cles))
print("Nombre de stations dans le JSON :", len(stations_obj))

# Dictionnaire de types fixés explicitement (pas inferSchema).
string_fields = ["station_id", "id", "name", "station_name", "city", "quartier",
                 "neighborhood", "district", "zone_type", "type_zone", "zone",
                 "zone_category", "type_de_zone"]
int_fields = ["capacity", "capacity_docks", "dock_capacity", "nb_docks", "capacity_bornes"]
double_fields = ["latitude", "longitude", "lat", "lon", "lng"]
fields = ([StructField(k, StringType(), True) for k in string_fields if k in cles]
          + [StructField(k, IntegerType(), True) for k in int_fields if k in cles]
          + [StructField(k, DoubleType(), True) for k in double_fields if k in cles])
# Préserver aussi les éventuels champs supplémentaires, explicitement en string.
fields += [StructField(k, StringType(), True) for k in sorted(cles)
           if k not in set(string_fields + int_fields + double_fields)]
schema_stations = StructType(fields)

# Lire toutes les stations directement depuis les objets JSON validés,
# avec le schéma explicite (compatible tableau JSON et JSONL).
# Pour éviter toute inférence implicite, on normalise en lignes JSONL partagées.
normalise = Path("/workspace/data/stations_normalized.jsonl")
with normalise.open("w", encoding="utf-8") as fh:
    for station in stations_obj:
        fh.write(json.dumps(station, ensure_ascii=False) + "\n")

df_stations = spark.read.schema(schema_stations).json("file:///workspace/data/stations_normalized.jsonl")
print("Stations chargées :", df_stations.count())
df_stations.printSchema()
df_stations.show(3, truncate=False)
assert df_stations.count() == 400, "Le professeur attend 400 stations"

Champs stations : ['capacity', 'city', 'latitude', 'longitude', 'name', 'quartier', 'station_id', 'zone_type']
Nombre de stations dans le JSON : 400
Stations chargées : 400
root
 |-- station_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- quartier: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- capacity: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

+----------+--------------------+----------+----------+-----------+--------+-----------------+-------------------+
|station_id|name                |city      |quartier  |zone_type  |capacity|latitude         |longitude          |
+----------+--------------------+----------+----------+-----------+--------+-----------------+-------------------+
|S-CAS-001 |Station Casablanca 1|Casablanca|Quartier 1|gare       |30      |33.54200648545325|-7.5704759600192135|
|S-CAS-002 |Station Casablanca 2|Casablanca|

### Normaliser les colonnes nécessaires à la jointure

In [7]:
def premier_existant(candidats, disponibles, label):
    for c in candidats:
        if c in disponibles:
            return c
    raise KeyError(f"Champ {label} introuvable. Disponibles : {sorted(disponibles)}")

station_key = premier_existant(["station_id", "id"], cles, "identifiant station")
zone_key = premier_existant(["zone_type", "type_zone", "zone_category", "type_de_zone", "zone"], cles, "type de zone")

stations_ref = (df_stations
    .select(F.col(station_key).cast("string").alias("station_id"),
            F.col(zone_key).cast("string").alias("zone_type"),
            *( [F.col("city").alias("station_city")] if "city" in cles else [] ))
    .dropDuplicates(["station_id"]))

# Joindre sur start_station ; left pour détecter les stations manquantes.
df_joint = (df_ended.alias("t")
    .join(F.broadcast(stations_ref).alias("s"),
          F.col("t.start_station") == F.col("s.station_id"), "left"))

nb_sans_station = df_joint.filter(F.col("zone_type").isNull()).count()
print("Trajets ENDED non associés à une station :", nb_sans_station)
print("Lignes après jointure :", df_joint.count())
assert df_joint.count() == df_ended.count()

df_zone_ville = (df_joint
    .groupBy("city", "zone_type")
    .agg(F.count("*").alias("nb_trajets"),
         F.round(F.sum("price_mad"), 2).alias("ca_mad"))
    .orderBy("city", "zone_type"))
print("=== TRAJETS ET CA PAR VILLE / TYPE DE ZONE ===")
df_zone_ville.show(100, truncate=False)

Trajets ENDED non associés à une station : 0
Lignes après jointure : 30228
=== TRAJETS ET CA PAR VILLE / TYPE DE ZONE ===


[Stage 85:=============================>                            (1 + 1) / 2]

+----------+---------------+----------+--------+
|city      |zone_type      |nb_trajets|ca_mad  |
+----------+---------------+----------+--------+
|Casablanca|centre_affaires|1779      |40553.32|
|Casablanca|gare           |1015      |23480.1 |
|Casablanca|plage          |933       |21686.57|
|Casablanca|residentiel    |1863      |43210.0 |
|Casablanca|tourisme       |960       |21937.16|
|Casablanca|universite     |1025      |23830.94|
|Marrakech |centre_affaires|1758      |40167.31|
|Marrakech |gare           |923       |20943.86|
|Marrakech |plage          |916       |21097.14|
|Marrakech |residentiel    |1748      |40266.19|
|Marrakech |tourisme       |1294      |29810.58|
|Marrakech |universite     |904       |20749.24|
|Rabat     |centre_affaires|1739      |40651.32|
|Rabat     |gare           |1423      |32773.38|
|Rabat     |plage          |878       |19189.09|
|Rabat     |residentiel    |1713      |38444.96|
|Rabat     |tourisme       |892       |19990.1 |
|Rabat     |universi

## 5. UDF `trip_band` contre `when/otherwise`

In [8]:
from pyspark.sql.types import StringType

# Critères du professeur : court <10 min, moyen de 10 à 30 min, long >30 min.
def trip_band(secondes):
    if secondes is None:
        return None
    if secondes < 600:
        return "court"
    if secondes <= 1800:
        return "moyen"
    return "long"

trip_band_udf = F.udf(trip_band, StringType())

avec_udf = df_ended.withColumn("trip_band", trip_band_udf(F.col("duration_s")))
avec_native = df_ended.withColumn(
    "trip_band",
    F.when(F.col("duration_s") < 600, "court")
     .when(F.col("duration_s") <= 1800, "moyen")
     .otherwise("long")
)

print("=== REPARTITION PAR TRIP_BAND ===")
avec_native.groupBy("trip_band").count().orderBy("trip_band").show()

# Les deux implémentations doivent donner les mêmes résultats.
repartition_udf = {r["trip_band"]: r["count"] for r in avec_udf.groupBy("trip_band").count().collect()}
repartition_native = {r["trip_band"]: r["count"] for r in avec_native.groupBy("trip_band").count().collect()}
assert repartition_udf == repartition_native
print("RESULTATS IDENTIQUES :", repartition_native)

=== REPARTITION PAR TRIP_BAND ===


+---------+-----+
|trip_band|count|
+---------+-----+
|    court| 3711|
|     long|15852|
|    moyen|10665|
+---------+-----+



RESULTATS IDENTIQUES : {'long': 15852, 'moyen': 10665, 'court': 3711}


In [ ]:
# L'agrégation oblige Spark à calculer réellement trip_band.
requete_udf = avec_udf.groupBy("trip_band").count()
requete_native = avec_native.groupBy("trip_band").count()
t_udf = chrono(lambda: requete_udf.collect())
t_native = chrono(lambda: requete_native.collect())
print("UDF Python (s)  :", [round(x, 3) for x in t_udf],
      "; médiane =", round(statistics.median(t_udf), 3))
print("when/otherwise :", [round(x, 3) for x in t_native],
      "; médiane =", round(statistics.median(t_native), 3))
print("EXPLAIN UDF :")
requete_udf.explain(True)
print("EXPLAIN NATIF :")
requete_native.explain(True)
print("Commentaire : une UDF Python ajoute des échanges JVM/Python (BatchEvalPython) ;")
print("les fonctions natives sont directement optimisables par Catalyst.")

## 6. `explain(True)` — jointure et Catalyst

In [9]:
print("=== EXPLAIN(TRUE) : JOINTURE + AGREGATION ===")
df_zone_ville.explain(True)
print("\nA COMMENTER DANS LE RAPPORT :")
print("1. Parsed Logical Plan ; 2. Analyzed Logical Plan ; 3. Optimized Logical Plan ;")
print("4. Physical Plan : BroadcastHashJoin / SortMergeJoin selon les ressources et le plan.")
print("Le plan physique observe (pas suppose) est la preuve de la strategie appliquee.")

=== EXPLAIN(TRUE) : JOINTURE + AGREGATION ===
== Parsed Logical Plan ==
'Sort ['city ASC NULLS FIRST, 'zone_type ASC NULLS FIRST], true
+- Aggregate [city#6, zone_type#321], [city#6, zone_type#321, count(1) AS nb_trajets#491L, round(sum(price_mad#11), 2) AS ca_mad#493]
   +- Join LeftOuter, (start_station#4 = station_id#320)
      :- SubqueryAlias t
      :  +- Filter ((status#7 = ENDED) AND isnotnull(price_mad#11))
      :     +- Relation [trip_id#0,user_id#1,vehicle_id#2,vehicle_type#3,start_station#4,end_station#5,city#6,status#7,duration_s#8,distance_km#9,battery_pct#10,price_mad#11,event_time#12] json
      +- SubqueryAlias s
         +- ResolvedHint (strategy=broadcast)
            +- Deduplicate [station_id#320]
               +- Project [cast(station_id#236 as string) AS station_id#320, cast(zone_type#240 as string) AS zone_type#321, city#238 AS station_city#322]
                  +- Relation [station_id#236,name#237,city#238,quartier#239,zone_type#240,capacity#241,latitude#242

## 7. Couche Silver — Parquet partitionné par `city`

In [10]:
# Le volume /workspace est monté dans Spark Master, les Workers et Jupyter.
# Écriture locale partagée, puis transfert vers le bucket MinIO lake.
SORTIE_PARQUET = "/workspace/data/silver/trips_history"

df_silver = (avec_native
    .filter(F.col("trip_id").isNotNull() & F.col("city").isNotNull())
    .withColumn("event_timestamp", F.to_timestamp("event_time")))

(df_silver.write
   .mode("overwrite")
   .partitionBy("city")
   .parquet("file://" + SORTIE_PARQUET))

print("Sortie Parquet locale :", SORTIE_PARQUET)
for dossier in sorted(Path(SORTIE_PARQUET).glob("city=*")):
    print(dossier.name, ":", len(list(dossier.glob("*.parquet"))), "fichier(s)")

df_relecture = spark.read.parquet("file://" + SORTIE_PARQUET)
print("Trajets Silver relus :", df_relecture.count())
assert df_relecture.count() == df_silver.count()
print("VALIDATION : PARQUET LISIBLE")

Sortie Parquet locale : /workspace/data/silver/trips_history
city=Casablanca : 2 fichier(s)
city=Marrakech : 2 fichier(s)
city=Rabat : 2 fichier(s)
city=Tanger : 2 fichier(s)


Trajets Silver relus : 30228
VALIDATION : PARQUET LISIBLE


## 8. Transférer les Parquet dans MinIO (`lake/silver/trips_history/`)

**À exécuter seulement après validation des cellules précédentes.** Le code suivant utilise uniquement la bibliothèque standard Python et une signature AWS SigV4 ; **aucune installation supplémentaire** n'est requise. Les identifiants sont demandés à l'exécution et ne sont pas enregistrés dans le notebook. Le transfert ne fonctionne que si `pedali-minio-test` est actif sur `pedali-test`.

Si tu ne souhaites pas saisir les identifiants dans le notebook, tu peux téléverser les fichiers depuis la console MinIO, en conservant le préfixe `silver/trips_history/`.

In [11]:
from getpass import getpass
from urllib.request import Request, urlopen
from urllib.parse import quote
from urllib.error import HTTPError
from datetime import datetime, timezone
import hashlib, hmac, mimetypes

MINIO_ENDPOINT = "http://pedali-minio-test:9000"
BUCKET = "lake"
PREFIX = "silver/trips_history/"

access_key = input("MinIO Access Key (locale) : ").strip()
secret_key = getpass("MinIO Secret Key (ne sera pas affichee) : ")
assert access_key and secret_key

def signer_cle(secret, date, region="us-east-1", service="s3"):
    def signe(k, m):
        return hmac.new(k, m.encode("utf-8"), hashlib.sha256).digest()
    k_date = signe(("AWS4" + secret).encode("utf-8"), date)
    k_region = signe(k_date, region)
    k_service = signe(k_region, service)
    return signe(k_service, "aws4_request")

def upload_sigv4(chemin_fichier, objet):
    data = chemin_fichier.read_bytes()
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    date = timestamp[:8]
    h_data = hashlib.sha256(data).hexdigest()
    # canonical URI /bucket/key ; encode each path segment
    uri = "/" + quote(BUCKET, safe="") + "/" + "/".join(
        quote(part, safe="-_.~") for part in objet.split("/"))
    host = "pedali-minio-test:9000"
    canonical_headers = (f"host:{host}\n"
                         f"x-amz-content-sha256:{h_data}\n"
                         f"x-amz-date:{timestamp}\n")
    signed_headers = "host;x-amz-content-sha256;x-amz-date"
    canonical_request = "\n".join([
        "PUT", uri, "", canonical_headers, signed_headers, h_data
    ])
    scope = f"{date}/us-east-1/s3/aws4_request"
    to_sign = "\n".join([
        "AWS4-HMAC-SHA256", timestamp, scope,
        hashlib.sha256(canonical_request.encode("utf-8")).hexdigest()
    ])
    sig = hmac.new(signer_cle(secret_key, date),
                   to_sign.encode("utf-8"), hashlib.sha256).hexdigest()
    auth = (f"AWS4-HMAC-SHA256 Credential={access_key}/{scope}, "
            f"SignedHeaders={signed_headers}, Signature={sig}")
    req = Request(MINIO_ENDPOINT + uri, data=data, method="PUT", headers={
        "Authorization": auth,
        "x-amz-date": timestamp,
        "x-amz-content-sha256": h_data,
        "Content-Type": mimetypes.guess_type(str(chemin_fichier))[0] or "application/octet-stream"
    })
    with urlopen(req, timeout=90) as response:
        return response.status

racine = Path(SORTIE_PARQUET)
fichiers = sorted(p for p in racine.rglob("*") if p.is_file() and
                  (p.suffix == ".parquet" or p.name in ["_SUCCESS", ".crc"]))
fichiers = [p for p in fichiers if p.suffix == ".parquet" or p.name == "_SUCCESS"]
assert fichiers, "Aucun fichier Parquet local trouvé"
for fichier in fichiers:
    cle = PREFIX + fichier.relative_to(racine).as_posix()
    statut = upload_sigv4(fichier, cle)
    print(statut, f"s3://{BUCKET}/{cle}")
print("TRANSFERT TERMINE - ouvrir la console MinIO et vérifier les 4 partitions city=...")
# Ne pas conserver les secrets dans les variables après l'envoi.
secret_key = None
access_key = None

MinIO Access Key (locale) :  pedaliadmin
MinIO Secret Key (ne sera pas affichee) :  ········


200 s3://lake/silver/trips_history/_SUCCESS
200 s3://lake/silver/trips_history/city=Casablanca/part-00000-2fbc36fd-6138-4818-bf4b-be4a37ee5796.c000.snappy.parquet
200 s3://lake/silver/trips_history/city=Casablanca/part-00001-2fbc36fd-6138-4818-bf4b-be4a37ee5796.c000.snappy.parquet
200 s3://lake/silver/trips_history/city=Marrakech/part-00000-2fbc36fd-6138-4818-bf4b-be4a37ee5796.c000.snappy.parquet
200 s3://lake/silver/trips_history/city=Marrakech/part-00001-2fbc36fd-6138-4818-bf4b-be4a37ee5796.c000.snappy.parquet
200 s3://lake/silver/trips_history/city=Rabat/part-00000-2fbc36fd-6138-4818-bf4b-be4a37ee5796.c000.snappy.parquet
200 s3://lake/silver/trips_history/city=Rabat/part-00001-2fbc36fd-6138-4818-bf4b-be4a37ee5796.c000.snappy.parquet
200 s3://lake/silver/trips_history/city=Tanger/part-00000-2fbc36fd-6138-4818-bf4b-be4a37ee5796.c000.snappy.parquet
200 s3://lake/silver/trips_history/city=Tanger/part-00001-2fbc36fd-6138-4818-bf4b-be4a37ee5796.c000.snappy.parquet
TRANSFERT TERMINE - ouvr

In [12]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Exemple de classement des trajets selon leur durée
def classer_trajet(duree):
    if duree is None:
        return "INCONNU"
    if duree < 600:
        return "COURT"
    if duree < 1800:
        return "MOYEN"
    return "LONG"

udf_classement = F.udf(classer_trajet, StringType())

df_comparaison = (
    df_trips
    .withColumn(
        "classe_udf",
        udf_classement(F.col("duration_s"))
    )
    .withColumn(
        "classe_native",
        F.when(F.col("duration_s").isNull(), "INCONNU")
         .when(F.col("duration_s") < 600, "COURT")
         .when(F.col("duration_s") < 1800, "MOYEN")
         .otherwise("LONG")
    )
)

erreurs = df_comparaison.filter(
    ~F.col("classe_udf").eqNullSafe(F.col("classe_native"))
).count()

print("=== COMPARAISON UDF VS FONCTIONS NATIVES ===")
print("Nombre de differences :", erreurs)

assert erreurs == 0
print("VALIDATION : RESULTATS IDENTIQUES")


[Stage 108:>                                                        (0 + 2) / 2]

=== COMPARAISON UDF VS FONCTIONS NATIVES ===
Nombre de differences : 0
VALIDATION : RESULTATS IDENTIQUES


In [13]:

print("=== VERIFICATION PARQUET SILVER ===")

# Lecture des Parquet exportes localement
df_silver = spark.read.parquet(
    "file:///workspace/data/silver/trips_history"
)

print("Nombre de lignes Silver :", df_silver.count())
print("Nombre de villes :", df_silver.select("city").distinct().count())

df_silver.groupBy("city").count().orderBy("city").show()

print("SCHEMA PARQUET :")
df_silver.printSchema()


=== VERIFICATION PARQUET SILVER ===
Nombre de lignes Silver : 30228


Nombre de villes : 4
+----------+-----+
|      city|count|
+----------+-----+
|Casablanca| 7575|
| Marrakech| 7543|
|     Rabat| 7562|
|    Tanger| 7548|
+----------+-----+

SCHEMA PARQUET :
root
 |-- trip_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- start_station: string (nullable = true)
 |-- end_station: string (nullable = true)
 |-- status: string (nullable = true)
 |-- duration_s: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- battery_pct: integer (nullable = true)
 |-- price_mad: double (nullable = true)
 |-- event_time: string (nullable = true)
 |-- trip_band: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- city: string (nullable = true)



## 9. Synthèse pour le rapport

- **Schéma explicite** : `StructType` pour les événements et les stations (sans `inferSchema`).
- **RDD / DataFrame / Spark SQL** : comparer les résultats identiques de CA par ville. RDD non typé au niveau des champs, DataFrame avec schéma et optimisation Catalyst ; Dataset fortement typé principalement en Scala/Java, pas d’API Dataset typée équivalente en PySpark.
- **Jointure** : type de zone des stations de départ, KPI par ville et zone.
- **UDF vs natif** : la version `when/otherwise` est optimisable directement par Catalyst ; une UDF Python implique un traitement séparé JVM/Python. Les temps mesurés dépendent de l’environnement et doivent être interprétés prudemment.
- **Catalyst / Tungsten** : Catalyst analyse et optimise les plans SQL ; Tungsten vise l’efficacité mémoire/CPU de l’exécution.
- **Parquet** : couche Silver partitionnée par `city`, dans MinIO à `lake/silver/trips_history/` après transfert.

**Captures à envoyer pour le PDF :** résultats DataFrame + SQL ; agrégation par ville/type de zone ; `explain(True)` (plans) ; console MinIO avec les dossiers `city=...`.